## Model Problem A: Gaussian Targets

Assuming that 
$$
    \mu_0 = N(0, C) \quad \mu = N(m, \tilde{C})
$$
and then we can write
$$
    \mu(dx) \propto \exp( - \Phi(x)) \mu_0(dx)
$$
where 
$$
    \Phi(x) = \frac{1}{2} \langle (\tilde{C}^{-1} + C^{-1})x, x \rangle  
    - \langle \tilde{C}^{-1} x , m \rangle.
$$

## Model Problem B: Find the Surface

We consider the statistical inversion problem of estimating $\mathbf{x} \in \mathbb{R}^n$ from data $z \in \mathbb{R}$ gathered according the measurement model:
$$
    z = f(\mathbf{x}) + \eta     \tag{MM1}
$$    
where
$$    
    \eta \sim N(0, \sigma^2) \text{ and } f: \mathbb{R}^n \to \mathbb{R}
    \text{ is the forward function. }
$$
Leaving aside the measurement noise $\eta$ we would like to recover the preimage $f^{-1}(z)$.  We constrain this set with Gaussian prior $\mu_0 \sim N(0, C)$ on $\mathbf{x}$. Then the posterior 'bayesian solution' $\mathbf{x}|z$ is the distribution:
$$
    \mu(d\mathbf{x}) \propto \exp\left( - \frac{1}{2 \sigma^2} ( f(\mathbf{x}) - z)^2 
    - \frac{1}{2} <C^{-1}\mathbf{x}, \mathbf{x}>\right) d\mathbf{x}.
    \tag{S1}
$$

## Model Problem C: Matrix Coefficents from Solutions

We next consider inverse problem of estimating the coefficents of an $n \times n$ anti-semetric matrix $A$ from data $\theta \in \mathbb{R}^n$ gathered according the measurement model:
$$
    z = \mathcal{O}(\theta(A)) + \eta \qquad \text{ where } \eta \sim N(0, \sigma^2)
    \tag{MM2}
$$
Here $\theta := \theta(A)$ is the solution of
$$
    (A + \kappa I)\theta = g
$$
and where $\mathcal{O}(\theta)$ defined by $\mathcal{O}: \mathbb{R}^n \to \mathbb{R}^m$ is a partial observation of $\theta$. Typically
$$
    \mathcal{O}((\theta_1, \ldots, \theta_n)) = (\theta_1,\ldots, \theta_m)
$$
Here the Bayesian posterior will take the form
$$
    \mu(dA) \propto \exp\left( - \frac{1}{2 \sigma^2} 
    | \mathcal{O}( (A +\kappa I)^{-1} g) - z|^2 
    - \frac{1}{2} <C^{-1}A, A>\right) dA.
    \tag{S1}
$$
which is a target in $d = n(n-1)/2$






To Do:

*Add experiments of type A for the mean shifts
*Add experiments for AD with stranger observations
*Add high dimensional AD experiment.

In [5]:
%load_ext autoreload
%autoreload 2

#Set-up: Packages to Import and Untility Functions

#Core Numerical Packages for Paral
import MCMC_Sampliers_Testing as MCMCsmp
import Utilities as Utils

#Numerical Elements
from numpy.linalg import norm
import numpy as np
from numpy import dot, array, transpose, diag
import random
import math

#Input Output utils
import os
import pandas as pd

#Stats elements
#from scipy.stats import norm
from scipy.stats import chi2
rng = np.random.default_rng()

#Plotting stuff
from matplotlib.patches import Ellipse

#Finished Message
import smtplib
from email.message import EmailMessage

from functools import partial
import multiprocessing as mp

saveLocBase = "/Users/negh_macpro/Library/CloudStorage/Dropbox/MCMC_Runs/" #MacbookPro
saveLocBaseData = "/Users/negh_macpro/Library/CloudStorage/Dropbox/MCMC_Runs/Data/"

def run_p_Sweep_Batches(ImpLists, q0Fn, TarDim, CovPrior, Pot, saveFileBase,saveFileBaseData,tsComp,burnIn):
    for ImpList in ImpLists:
        curPList = [I[0] for I in ImpList]
        print("Running Blocks for: " + str(curPList))
        saveDictLocCur = saveFileBaseData + "data_dict_info_p_"
        for p in curPList:
           saveDictLocCur += str(p) + "_" 
        saveDictLocCur += ".npy"
        if __name__ == "__main__":
            par_sweep_dict = Utils.parameter_sweep_p_rho(ImpList, q0FN, TarDim, CovPrior, Pot, saveDictExt=True,
                                                           saveDictLoc = saveDictLocCur)

        #par_sweep_dictEx1 = np.load(saveDictLocCur, allow_pickle=True).item()

        Utils.parameter_sweep_p_rho_save_figures(par_sweep_dict, TarDim, tsComp,burnIn, saveFileBase)


#FORMAT: [p,NumRho,NumSampsESS,numChainsESS, NumChainsgM]
#            -p: value to p to run study
#            -NumRho: 1/NumRho specfies the step size in rho over [0,1] for the study
#            -NumSampsESS: Length of the MCMC at each rho value to compute ESS/MSJD
#            -numChainsESS: Number of independent chains to compute ESS/MSJD
#            -NumChainsgM: Number of separate chain M= NumChainsgM to compute Var(\bar{g}_N^\rho)

#Test Run
ImpList1 = [[3,10,5000,5,5], [7,10,5000,5,5]]
ImpList2 = [[9,10,5000,5,5], [12,10,5000,5,5]]
ImpLists = [ImpList1, ImpList2]

#Hardcore Run
#ImpList1 = [[5,50,50000,100,5000], [10,50,50000,100,5000], [20,50,50000,100,5000], [40,50,50000,100,5000]]
#ImpList2 = [[80,50,50000,100,5000], [160,50,50000,100,5000]]
#ImpList3 = [[320,50,50000,100,5000], [640,50,50000,100,5000]]
#ImpList4 = [[1280,50,50000,100,5000]]
#ImpLists = [ImpList1, ImpList2, ImpList3,ImpList4]
#ImpList5 = [[5000,50,50000,100,5000]]
#ImpList6 = [[10000,50,50000,100,5000]]
#ImpLists = [ImpList5,ImpList6]


def make_stat_measure(TarDim, priorCov,Pot, chainLn =1000, numChain =10,  MC_meth =MCMCsmp.MpCN, warmrho = .2, warmp = 1000,burnIn=500,thinfactor=5):
    q0gen = lambda: np.random.multivariate_normal(np.zeros(TarDim), priorCov)
    MC_arg = [TarDim,priorCov,warmrho,Pot,warmp]
    samps, ESS = Utils.parallel_MCMC_Runs_ESS(chainLn,numChain,MC_meth,MC_arg,q0gen, burnIn)

    ESS_N_min = ESS.min()/(chainLn*numChain)
    print(ESS_N_min)
    thin_smp = thinfactor*int(1/ESS_N_min)
    print(thin_smp)
    thinned_samps = samps[::thin_smp]
    q0genFull = lambda:random.choice(thinned_samps)

    #parallel_MCMC_Runs(chainLn,numChain,MCMCmeth,MCMCmethArgs, q0gen, burn_In, thin)
    return Utils.parallel_MCMC_Runs(chainLn,numChain*thin_smp,MC_meth,MC_arg,q0genFull, burnIn,thinfactor)

print("Current cpu count:" + str(mp.cpu_count()))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Current cpu count:11


In [ ]:
#Experiment A1
#Gaussian target-- Shifted Covariance on `low modes'

#Make Problem Information File
expId =  1
ExpDir = "Large_p_study/Experiment_A/" + "Ex_ID_"+ str(expId) + "/"
FileNmBase = saveLocBase+ExpDir
FileNmBaseData =saveLocBaseData+ExpDir
os.makedirs(FileNmBase, exist_ok=True)
os.makedirs(FileNmBaseData, exist_ok=True)


#Model dim
TarDimExA1 = 5
#Covariance algebraic decay
cov0 = 4
gam = 2
CovDiag_p = [cov0* (j**(-gam)) for j in list(range(1,TarDimExA1+1))]

CovPriorExA1 = Utils.mkDiagCov(CovDiag_p)
print("Prior Covariance:")
print(CovPriorExA1)

#Information for Gaussian Posterior
#Perturb dim
pertDimExA1 = 2
CovDiag_post = [CovDiag_p[1], CovDiag_p[0]] + CovDiag_p[pertDimExA1:]
CovPostExA1 = Utils.mkDiagCov(CovDiag_post)
meanPostExA1 = np.zeros(TarDimExA1)

print("Posterior Mean:")
print(meanPostExA1)
print("Posterior Covariance:")
print(CovPostExA1)

infoStr  = "Target Type:  Model Problem A -- Gaussian Target \n"
infoStr += "Target Dimension: " + str(TarDimExA1) + "\n"
infoStr += "Prior Covariance: \n" + str(CovPriorExA1) + "\n"
infoStr += "Perturbation Dimension: " + str(pertDimExA1) + "\n"
infoStr += "Posterior Mean: " + str(meanPostExA1) + "\n"
infoStr += "Posterior Covariance: \n" + str(CovPostExA1)+ "\n"

Utils.message_ntfy("RUNNING EXAMPLE A1")
Utils.message_ntfy(infoStr)

probDataFile = FileNmBase + "Problem_Info.txt"
with open(probDataFile, 'w') as file:
    file.write(infoStr)

PriorCovInvExA1 = Utils.mkDiagCov([CovDiag_p[0]**(-1),CovDiag_p[1]**(-1)])
PostCovInvExA1 = Utils.mkDiagCov([CovDiag_post[0]**(-1),CovDiag_post[1]**(-1)])

PotExA1Pass = partial(Utils.PotGaussPert, TarDim=TarDimExA1, PertDim=pertDimExA1, PriorCovInv=PriorCovInvExA1, PostMean=meanPostExA1[0:pertDimExA1], PostCovInv=PostCovInvExA1, mode = "soft")


q0FN = lambda nmICs: np.random.multivariate_normal(np.zeros(TarDimExA1), CovPostExA1, size=nmICs)


run_p_Sweep_Batches(ImpLists, q0FN, TarDimExA1, CovPriorExA1, PotExA1Pass, FileNmBase, FileNmBaseData,[0,1,4],1000)




In [ ]:
#Experiment A2
#Gaussian target-- Shifted Covariance on `low modes'--Higher Dimension

#Make Problem Information File
expId =  2
ExpDir = "Large_p_study/Experiment_A/" + "Ex_ID_"+ str(expId) + "/"
FileNmBase = saveLocBase+ExpDir
FileNmBaseData =saveLocBaseData+ExpDir
os.makedirs(FileNmBase, exist_ok=True)
os.makedirs(FileNmBaseData, exist_ok=True)


#Model dim
TarDimExA2 = 20
#Covariance algebraic decay
cov0 = 4
gam = 2
CovDiag_p = [cov0* (j**(-gam)) for j in list(range(1,TarDimExA2+1))]

CovPriorExA2 = Utils.mkDiagCov(CovDiag_p)
print("Prior Covariance:")
print(TarDimExA2)

#Information for Gaussian Posterior
#Perturb dim
pertDimExA2 = 2
CovDiag_post = [CovDiag_p[1], CovDiag_p[0]] + CovDiag_p[pertDimExA2:]
CovPostExA2 = Utils.mkDiagCov(CovDiag_post)
meanPostExA2 = np.zeros(TarDimExA2)

print("Posterior Mean:")
print(meanPostExA2)
print("Posterior Covariance:")
print(CovPostExA2)

infoStr  = "Target Type:  Model Problem A -- Gaussian Target \n"
infoStr += "Target Dimension: " + str(TarDimExA2) + "\n"
infoStr += "Prior Covariance: \n" + str(CovPriorExA2) + "\n"
infoStr += "Perturbation Dimension: " + str(pertDimExA2) + "\n"
infoStr += "Posterior Mean: " + str(meanPostExA2) + "\n"
infoStr += "Posterior Covariance: \n" + str(CovPostExA2)+ "\n"

Utils.message_ntfy("RUNNING EXAMPLE A2")
Utils.message_ntfy(infoStr)

probDataFile = FileNmBase + "Problem_Info.txt"
with open(probDataFile, 'w') as file:
    file.write(infoStr)

PriorCovInvExA2 = Utils.mkDiagCov([CovDiag_p[0]**(-1),CovDiag_p[1]**(-1)])
PostCovInvExA2 = Utils.mkDiagCov([CovDiag_post[0]**(-1),CovDiag_post[1]**(-1)])

PotExA2Pass = partial(Utils.PotGaussPert, TarDim=TarDimExA2, PertDim=pertDimExA2, PriorCovInv=PriorCovInvExA2, PostMean=meanPostExA2[0:pertDimExA2], PostCovInv=PostCovInvExA2, mode = "soft")


q0FN = lambda nmICs: np.random.multivariate_normal(np.zeros(TarDimExA2), CovPostExA2, size=nmICs)


run_p_Sweep_Batches(ImpLists, q0FN, TarDimExA2, CovPriorExA2, PotExA2Pass, FileNmBase,FileNmBaseData,[0,1,4],1000)



In [6]:
#Experiment B1

%load_ext autoreload
%autoreload 2

#Set-up: Packages to Import and Untility Functions

#Core Numerical Packages for Paral
import MCMC_Sampliers_Testing as MCMCsmp
import Utilities as Utils

#We consider a target of the type Model Problem B, [MM1] where
#    f(x,y) = y(x-a)^r 

#Make Problem Information File
expId =  1
ExpDir = "Large_p_study/Experiment_B/" + "Ex_ID_"+ str(expId) + "/"
FileNmBase = saveLocBase+ExpDir
FileNmBaseData =saveLocBaseData+ExpDir
os.makedirs(FileNmBase, exist_ok=True)
os.makedirs(FileNmBaseData, exist_ok=True)

NumParmsExB1 = 2
zExB1 = 6
aExB1 = .3
rExB1 = 1
sigExB1 = 1
CovExB1 = np.diag([3,2])

PotExB1Pass = partial(Utils.PotExB1, sig=sigExB1, a= aExB1, r=rExB1, z=zExB1,  mode="soft")

probDataFile = FileNmBase + "Problem_Info.txt"
fFnStr = "y (x - " + str(aExB1) + ")^" + str(rExB1) 

with open(probDataFile, 'w') as file:
    file.write("Target Type:  Model Problem B \n")
    file.write("Model Dim: " + str(NumParmsExB1) + "\n")
    file.write("Forward Map:  f(x,y) = " + fFnStr + "\n")
    file.write("z: " + str(zExB1) + "\n")
    file.write("sig: " + str(sigExB1)+ "\n")
    file.write("Prior Cov: " + "\n" + str(CovExB1))

#make_stat_measure(TarDim, priorCov,Pot, chainLn =10000, numChain =100, MC_meth =MCMCsmp.MpCN, warmrho = .5, warmp = 200,burnIn=500,thinfactor=5):


FinalSampsExB1 = make_stat_measure(NumParmsExB1,CovExB1,PotExB1Pass)
numSampsExB1= FinalSampsExB1.shape[0]
print("Number of samples available: " + str(numSampsExB1))

statmeasureFnNm = FileNmBaseData + "stationary_seq.csv"
Utils.writeCSV(statmeasureFnNm, FinalSampsExB1)


R =5
dr=.1

histFileNm = FileNmBase + "Baseline_Histogram.png"
Utils.makeHistGrid(R, dr, FinalSampsExB1, NumParmsExB1,histFileNm, C=CovExB1, beta=0.95, hidePlt = True)



q0FN = lambda nmICs: FinalSampsExB1[rng.integers(0,numSampsExB1,size =nmICs)]

run_p_Sweep_Batches(ImpLists, q0FN, NumParmsExB1, CovExB1, PotExB1Pass, FileNmBase,FileNmBaseData,[0,1],1000)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Total MCMC Runs: 10


Parallel MCMC Runs:   0%|          | 0/10 [00:00<?, ?it/s]

0.3773324333534004
10
Total MCMC Runs: 100


Parallel MCMC Runs:   0%|          | 0/100 [00:00<?, ?it/s]

Number of samples available: 10100
Running Blocks for: [3, 7]
Currently running: p=3
Delta rho: 0.1
Number of Samples per Chain to compute ESS/MSJD: 5000
Number of Independent Chain to compute ESS/MSJD: 5
N in bar{g}_N^rho = 1/N sum_{j =1}^N g(X_jrho): 100
Number of separate chain M to compute Var(bar{g}_Nrho): 5
Total MCMC Runs Submitted: 27


...

Total Run Time Was: 0:00:06.465197


Building ESS Plots:   0%|          | 0/2 [00:00<?, ?it/s]

Building MSJD Plots:   0%|          | 0/2 [00:00<?, ?it/s]

Building Plots Var(g_N) for different g_N:   0%|          | 0/2 [00:00<?, ?it/s]

Building Time Series Plots:   0%|          | 0/2 [00:00<?, ?it/s]

In [11]:
#Experiment B2: Set-up

#We consider a target of the type Model Problem B, [MM1] where
#    f(x) = x* C^{-1} x

#The problem parameters are:
expId = 2 #Experiment ID
ExpDir = "Large_p_study/Experiment_B/" + "Ex_ID_"+ str(expId) + "/"
FileNmBase = saveLocBase+ExpDir
FileNmBaseData =saveLocBaseData+ExpDir
os.makedirs(FileNmBase, exist_ok=True)
os.makedirs(FileNmBaseData, exist_ok=True)

NumParmsExB2 = 2
cov0 = 4
gam = 2
CovDiag_p = [cov0* (j**(-gam)) for j in list(range(1,NumParmsExB2+1))]
CovInvDiag_p = [cov0**(-1) * (j**(gam)) for j in list(range(1,NumParmsExB2+1))]

sigExB2 = .5

#Compute zExB2 so that \mu_0({x | x^* C^{-1} x \leq zBxB2\}) = prob_below
prob_below = .95
zExB2 = chi2.ppf(prob_below, df=NumParmsExB2)  

CovPriorExB2 = Utils.mkDiagCov(CovDiag_p)
CovPriorInvExB2 = Utils.mkDiagCov(CovInvDiag_p) 


#Make Loglikehood function with lambda workaround
#using PotEx1(X,sig, a,r,z)
#print(inspect.signature(MCMCsmp.PotEx1))

#PotExB2(X, sig, CovInv, zdata)
PotExB2Pass = partial(Utils.PotMahalanobis, sig=sigExB2, CovInv = CovPriorInvExB2,  zdata = zExB2, mode="soft")




probDataFile = FileNmBase + "Problem_Info.txt"
fFnStrB2 = "x* C^{-1} x"

with open(probDataFile, 'w') as file:
    file.write("Target Type:  Model Problem B \n")
    file.write("Model Dim: " + str(NumParmsExB2) + "\n")
    file.write("Forward Map:  f(x) = " + fFnStrB2 + "\n")
    file.write("z: " + str(zExB2) + "\n")
    file.write("sig: " + str(sigExB2)+ "\n")
    file.write("Prior Cov: " + "\n" + str(CovPriorExB2))

#parallel_MCMC_Runs_ESS(chainLn,numChain,MCMCmeth,MCMCmethArgs, q0gen, burn_In):
#MpCN(q0,dim,Cov,rho,Pot,NProps,L)

FinalSampsExB2 = make_stat_measure(NumParmsExB2,CovPriorExB2,PotExB2Pass)

statmeasureFnNm = FileNmBaseData + "stationary_seq.csv"
Utils.writeCSV(statmeasureFnNm, FinalSampsExB2)


numSampsExB2= FinalSampsExB1.shape[0]
print("Number of samples available: " + str(numSampsExB2))

histFileNm = FileNmBase + "Baseline_Histogram.png"
Utils.makeHistGrid(R, dr, FinalSampsExB1, NumParmsExB2,histFileNm, C=CovExB1, beta=0.95, hidePlt = True)

q0FNB2 = lambda nmICs: FinalSampsExB1[rng.integers(0,numSampsExB2,size =nmICs)]

run_p_Sweep_Batches(ImpLists, q0FNB2, NumParmsExB2, CovPriorExB2, PotExB2Pass, FileNmBase,FileNmBaseData,[0,1],1000)


R =5
dr=.1

histFileNm = FileNmBase + "Baseline_Histogram.png"

Utils.makeHistGrid(R, dr, FinalSampsExB2, NumParmsExB2,histFileNm, C=CovPriorExB2, beta=0.95, hidePlt = True)


Total MCMC Runs: 10


Parallel MCMC Runs:   0%|          | 0/10 [00:00<?, ?it/s]

0.38797813006439685
10
Total MCMC Runs: 100


Parallel MCMC Runs:   0%|          | 0/100 [00:00<?, ?it/s]

Number of samples available: 10100
Running Blocks for: [3, 7]
Currently running: p=3
Delta rho: 0.1
Number of Samples per Chain to compute ESS/MSJD: 5000
Number of Independent Chain to compute ESS/MSJD: 5
N in bar{g}_N^rho = 1/N sum_{j =1}^N g(X_jrho): 100
Number of separate chain M to compute Var(bar{g}_Nrho): 5
Total MCMC Runs Submitted: 27


...

Total Run Time Was: 0:00:08.831289


Building ESS Plots:   0%|          | 0/2 [00:00<?, ?it/s]

Building MSJD Plots:   0%|          | 0/2 [00:00<?, ?it/s]

Building Plots Var(g_N) for different g_N:   0%|          | 0/2 [00:00<?, ?it/s]

Building Time Series Plots:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
#Experiment B3

#We consider a target of the type Model Problem B, where
#    f(x) = \pi(x)* (pi C)^{-1} \pi(x)
#where \pi is the projection onto the first two dimensions

#The problem parameters are:
expId = 3 #Experiment ID
ExpDir = "Large_p_study/Experiment_B/" + "Ex_ID_"+ str(expId) + "/"
FileNmBase = saveLocBase+ExpDir
FileNmBaseData =saveLocBaseData+ExpDir
os.makedirs(FileNmBase, exist_ok=True)
os.makedirs(FileNmBaseData, exist_ok=True)

NumParmsExB3 = 5
cov0 = 4
gam = 2
CovDiag_p = [cov0* (j**(-gam)) for j in list(range(1,NumParmsExB3+1))]
CovInvDiag_p = [cov0**(-1) * (j**(gam)) for j in list(range(1,NumParmsExB3+1))]

sigExB2 = .5

#Compute zExB3 so that \mu_0({\pi(x) | \pi(x)^* C^{-1} \pi(x) \leq zBxB2\}) = prob_below
informedDirB3 = 2
prob_below = .95
zExB3 = chi2.ppf(prob_below, df=informedDirB3)  

CovPriorExB3 = Utils.mkDiagCov(CovDiag_p)
CovPriorInvExB3 = Utils.mkDiagCov(CovInvDiag_p) 


#Make Loglikehood function with lambda workaround
#using PotEx1(X,sig, a,r,z)
#print(inspect.signature(MCMCsmp.PotEx1))

#PotExB2(X, sig, CovInv, zdata)
PotExB3Pass = partial(Utils.PotMahalanobis, sig=sigExB3, CovInv = CovPriorInvExB3,  zdata = zExB2, mode="soft")
#FIX



probDataFile = FileNmBase + "Problem_Info.txt"
fFnStrB3 = "x* C^{-1} x"

with open(probDataFile, 'w') as file:
    file.write("Target Type:  Model Problem B \n")
    file.write("Model Dim: " + str(NumParmsExB3) + "\n")
    file.write("Forward Map:  f(x) = " + fFnStrB3 + "\n")
    file.write("z: " + str(zExB3) + "\n")
    file.write("sig: " + str(sigExB3)+ "\n")
    file.write("Prior Cov: " + "\n" + str(CovPriorExB3))

FinalSampsExB3 = make_stat_measure(NumParmsExB3,CovExB3,PotExB3Pass)

statmeasureFnNm = FileNmBaseData + "stationary_seq.csv"
Utils.writeCSV(statmeasureFnNm, FinalSampsExB2)


R =5
dr=.1

histFileNm = FileNmBase + "Baseline_Histogram.png"
Utils.makeHistGrid(R, dr, sampsExB2, NumParmsExB2,histFileNm, C=CovPriorExB2, beta=0.95, hidePlt = True)

In [14]:
%load_ext autoreload
%autoreload 2

#Core Numerical Packages for Paral
import MCMC_Sampliers_Testing as MCMCsmp
import Utilities as Utils


#Experiment C1: Set-up

#This experiment is based on Model Problem C
#We use precisely the example appearing in refer [1]


#The problem parameters are:
expId = 1 #Experiment ID
ExpDir = "Large_p_study/Experiment_C/" + "Ex_ID_"+ str(expId) + "/"
FileNmBase = saveLocBase+ExpDir
FileNmBaseData =saveLocBaseData+ExpDir
os.makedirs(FileNmBase, exist_ok=True)
os.makedirs(FileNmBaseData, exist_ok=True)


#We Specifying Problem Parameters as follows
#Model dimension and parameter space size are
ModDmExC1= 4
NumParmsExC1 = int(ModDmExC1*(ModDmExC1 -1)/2)


expId = 1 #Experiment ID
 
gvecExC1 = np.array([.1,0,5,2])
sigExC1 = 2
zExC1 = np.array([4.601,18.021,0,0])
dataDimExC1 = 2
kapExC1 = .05

# Covariance of the 'prior' C = cov0[1^{-gam}, 2^[-gam],..., N^{-gam}]
# where N is the number of parameters in the model NumParms = 6 
#for number of enetries that need to be specified in A
cov0 = 5
gam = 1.5
CovExC1 = np.diag([cov0* (j**(-gam)) for j in list(range(1,NumParmsExC1+1))])


probDataFile = FileNmBase + "Problem_Info.txt"
with open(probDataFile, 'w') as file:
    file.write("Target Type:  Model Problem C \n")
    file.write("Model Dim: " + str(NumParmsExC1) + "\n")
    file.write("g: " + str(gvecExC1) + "\n")
    file.write("z: " + str(zExC1[0:2]) + "\n")
    file.write("sig: " + str(sigExC1)+ "\n")
    file.write("kappa: " + str(kapExC1)+ "\n")
    file.write("Prior Cov: " + "\n" + str(CovExC1))

#Make Loglikehood function with lambda workaround
#using PotExAD(a, gvec, sig, ModDm, z, kap, dataDim, mode = None):
#print(inspect.signature(MCMCsmp.PotExAD))


PotExC1Pass = partial(Utils.PotExAD, gvec=gvecExC1, sig=sigExC1, ModDm=ModDmExC1, z=zExC1, kap=kapExC1, dataDim=dataDimExC1, mode = "soft")

FinalSampsExC1 = make_stat_measure(NumParmsExC1,CovExC1,PotExC1Pass)

statmeasureFnNm = FileNmBaseData + "stationary_seq.csv"
Utils.writeCSV(statmeasureFnNm, FinalSampsExC1 )




numSampsExC1= FinalSampsExC1.shape[0]
print("Number of samples available: " + str(numSampsExC1))

histFileNm = FileNmBase + "Baseline_Histogram.png"
Utils.makeHistGrid(R, dr, FinalSampsExC1, NumParmsExC1,histFileNm, C=CovExC1, beta=0.95, hidePlt = True)

q0FNC1 = lambda nmICs: FinalSampsExB1[rng.integers(0,numSampsExC1,size =nmICs)]

run_p_Sweep_Batches(ImpLists, q0FNC1, NumParmsExC1, CovExC1, PotExC1Pass, FileNmBase,FileNmBaseData,[0,1,4],1000)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Total MCMC Runs: 10


Parallel MCMC Runs:   0%|          | 0/10 [00:00<?, ?it/s]

0.3063677027092831
15
Total MCMC Runs: 150


Parallel MCMC Runs:   0%|          | 0/150 [00:00<?, ?it/s]

Number of samples available: 15150
Running Blocks for: [3, 7]
Currently running: p=3
Delta rho: 0.1
Number of Samples per Chain to compute ESS/MSJD: 5000
Number of Independent Chain to compute ESS/MSJD: 5
N in bar{g}_N^rho = 1/N sum_{j =1}^N g(X_jrho): 100
Number of separate chain M to compute Var(bar{g}_Nrho): 5
Total MCMC Runs Submitted: 27


Parallel MCMC Runs:   0%|          | 0/27 [00:00<?, ?it/s]

ValueError: could not broadcast input array from shape (2,) into shape (6,)